# 02 — Robustness pilot on 32 malicious records

**Run this after notebook 01 succeeds.** This notebook tests both supplied RLN checkpoints on the same 32 malicious CTU-Neris records, with a larger constrained-attack search than the eight-record smoke test.

**Use Runtime → Run all.** Leave the settings unchanged for the planned pilot. CPU is sufficient. If you already selected a GPU runtime, the notebook still works but deliberately evaluates on CPU; see the [GPU audit and pilot guide](https://github.com/Abdulstar/MyPhDThesis/blob/main/docs/PILOT32_GUIDE.md).

The existing notebook and earlier outputs are preserved. A separate `/content/MyPhDThesis_Pilot32` checkout and unique result folders are used. Download the ZIP from the final cell and share it for review.

This is feature-space evaluation of **pretrained models**. It does not train your proposed defense or replay network packets.


## 1. Settings and experiment record

Default budget: **32 malicious records, 10 gradient steps, 10 evolutionary generations, 50 offspring, scaled L2 radius 0.5, seed 0**. Selection uses labels and metadata validity, never either model's prediction. Both checkpoints receive the same selected rows; already-missed records are counted separately from new evasions.

The first run downloads the dataset and installs the pinned environment. Repeating cells creates fresh output names. Drive copying is optional and disabled by default.


In [ ]:
import datetime as dt
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import uuid

PILOT_RECORDS = 32 #@param {type:"integer"}
ATTACK_STEPS = 10 #@param {type:"integer"}
ATTACK_GENERATIONS = 10 #@param {type:"integer"}
ATTACK_OFFSPRING = 50 #@param {type:"integer"}
EPSILON = 0.5 #@param {type:"number"}
SEED = 0 #@param {type:"integer"}
SAVE_TO_DRIVE = False #@param {type:"boolean"}
DOWNLOAD_RESULTS = True #@param {type:"boolean"}

REPO_URL = "https://github.com/Abdulstar/MyPhDThesis.git"
REPO_REF = "main"
ROOT = Path(os.environ.get("PHD_WORKSPACE", "/content/MyPhDThesis_Pilot32")).resolve()
HOST_PYTHON = sys.executable
HOST_VERSION = sys.version
if not 1 <= PILOT_RECORDS <= 407:
    raise ValueError("PILOT_RECORDS must be between 1 and 407; keep 32 for the planned pilot.")
if min(ATTACK_STEPS, ATTACK_GENERATIONS, ATTACK_OFFSPRING) < 1 or EPSILON <= 0:
    raise ValueError("Attack budgets must be positive.")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(ROOT)], check=True)
elif not (ROOT / ".git").is_dir():
    raise RuntimeError(f"{ROOT} exists but is not a git checkout. Choose a new workspace.")
# Reuse existing work without git reset, checkout or pull; record exactly what ran.
for required in ["scripts/bootstrap.py", "baseline/run_baseline.py", "notebooks/02_Robustness_Pilot_32.ipynb"]:
    if not (ROOT / required).is_file():
        raise RuntimeError(f"Missing {required}. Use a fresh clone of the completed repository.")

def unique_name(prefix):
    return prefix + "_" + dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]

SESSION = ROOT / "runs" / unique_name("pilot32")
SESSION.mkdir(parents=True, exist_ok=False)
sha = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--verify", "HEAD"], text=True, capture_output=True)
project_commit = sha.stdout.strip() if sha.returncode == 0 else "uncommitted-workspace"
source_paths = [p for folder in ["baseline", "scripts"] for p in (ROOT / folder).glob("*") if p.is_file()]
source_paths += [ROOT / "notebooks/02_Robustness_Pilot_32.ipynb"]
source_hashes = {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_paths}
context = {"project_repository": REPO_URL, "project_commit": project_commit,
           "project_status": subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).splitlines(),
           "notebook_python": HOST_VERSION, "notebook_executable": HOST_PYTHON,
           "session_utc": dt.datetime.now(dt.timezone.utc).isoformat(), "source_sha256": source_hashes,
           "experiment": "32_record_robustness_pilot", "evaluation_device": "cpu",
           "pilot_records": PILOT_RECORDS, "steps": ATTACK_STEPS,
           "generations": ATTACK_GENERATIONS, "offspring": ATTACK_OFFSPRING,
           "epsilon_scaled_l2": EPSILON, "seed": SEED}
(SESSION / "session_context.json").write_text(json.dumps(context, indent=2) + "\n")
SNAPSHOT = SESSION / "source_snapshot"
for p in source_paths:
    target = SNAPSHOT / p.relative_to(ROOT)
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, target)

def run_command(command, log_name):
    log_path = SESSION / log_name
    # Preserve a prior log if a cell is repeated.
    if log_path.exists():
        log_path = SESSION / (unique_name(Path(log_name).stem) + ".log")
    with log_path.open("w") as log:
        log.write("Command: " + json.dumps([str(x) for x in command]) + "\n")
        log.flush()
        process = subprocess.Popen([str(x) for x in command], cwd=ROOT,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        code = process.wait()
    if code:
        raise RuntimeError(f"Command failed ({code}); see {log_path}. Fix the error, then rerun this cell.")

print("Notebook Python:", sys.version.split()[0])
print("Project commit:", project_commit)
print("Results folder:", SESSION)

# Log accelerator allocation without importing or changing Colab's PyTorch.
hardware = {"evaluation_device": "cpu", "platform": sys.platform,
            "reason": "Pinned constrained-attack pipeline is validated on CPU only."}
if shutil.which("nvidia-smi"):
    try:
        probe = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                                "--format=csv,noheader"], capture_output=True, text=True, timeout=15)
        hardware.update({"nvidia_smi_returncode": probe.returncode,
                         "allocated_gpu": probe.stdout.strip(), "probe_stderr": probe.stderr.strip()})
    except (subprocess.SubprocessError, OSError) as exc:
        hardware["probe_error"] = str(exc)
else:
    hardware["allocated_gpu"] = None
(SESSION / "hardware.json").write_text(json.dumps(hardware, indent=2) + "\n")
print("Evaluation device: CPU. Allocated GPU:", hardware.get("allocated_gpu") or "none detected")
print("An allocated GPU is not used by this pinned pilot. CPU runtime is sufficient.")


## 2. Isolated baseline environment

Python 3.8.20 and CPU PyTorch 1.12.1 run separately from Colab's notebook Python. The original pinned upstream code is preserved. No kernel restart or manual package installation is needed.


In [ ]:
run_command([HOST_PYTHON, ROOT / "scripts/bootstrap.py"], "setup.log")
PYTHON = ROOT / ".baseline-env/bin/python"
assert sys.executable == HOST_PYTHON and sys.version == HOST_VERSION
shutil.copy2(ROOT / ".cache/setup_report.json", SESSION / "setup_report.json")
print("Notebook interpreter unchanged. Baseline interpreter:", PYTHON)


## 3. Verify dataset and pretrained checkpoints

Download approximately 348 MB on a fresh checkout. Existing files are checked against pinned SHA-256 hashes and reused when valid. These are precomputed CTU features, not PCAP traffic.


In [ ]:
run_command([PYTHON, ROOT / "baseline/download_assets.py", "--assets", ROOT / "assets"], "download.log")
shutil.copy2(ROOT / "assets/download_manifest.json", SESSION / "download_manifest.json")


## 4. Run the paired pilot

This command evaluates the full 55,082-record clean test partition and attacks the selected 32 malicious records for each model. It does not rerun the earlier eight-record experiment. Keep the runtime connected until it finishes; the log is saved even if a command fails.


In [ ]:
PILOT_DIR = SESSION / unique_name("attack32_g10")
run_command([PYTHON, ROOT / "baseline/run_baseline.py", "--repo", ROOT / "vendor/tabularbench",
             "--assets", ROOT / "assets", "--out", PILOT_DIR, "--attack",
             "--n-attack", str(PILOT_RECORDS), "--steps", str(ATTACK_STEPS),
             "--generations", str(ATTACK_GENERATIONS), "--offspring", str(ATTACK_OFFSPRING),
             "--eps", str(EPSILON), "--seed", str(SEED)], "pilot32.log")
run_command([PYTHON, ROOT / "scripts/summarize_results.py", PILOT_DIR], "pilot32_summary.log")


## 5. Check results and returned perturbations

We verify the clean confusion matrices and that both checkpoints used identical row IDs. We distinguish returned changed rows, valid changes, records already missed and newly successful evasions. CAA may return the original input when it finds no acceptable successful candidate; a valid unchanged input is not a successful attack.

A clean-score mismatch is printed and saved so the export cell can still preserve the evidence. Investigate a mismatch before making research claims.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

report = json.loads((PILOT_DIR / "results.json").read_text())
display(pd.read_csv(PILOT_DIR / "summary.csv"))
expected_counts = {"default": (54587, 88, 9, 398), "madry": (54633, 42, 11, 396)}
reference_checks = {}
for name, expected in expected_counts.items():
    actual = tuple(report["models"][name]["clean"][k] for k in ["tn", "fp", "fn", "tp"])
    reference_checks[name] = {"actual_tn_fp_fn_tp": actual, "expected_tn_fp_fn_tp": expected,
                              "matches": actual == expected}
    print(name, "clean reference match:", actual == expected)
    if actual != expected:
        print("Investigate this mismatch before interpreting the attack comparison; keep the exported logs.")

rows = []
selected_rows = np.asarray(report["selection"]["global_csv_row_indices_zero_based"])
for name in ["default", "madry"]:
    attack = report["models"][name]["attack"]
    audit = pd.read_csv(PILOT_DIR / f"{name}_attack_audit.csv")
    with np.load(PILOT_DIR / f"{name}_attack_arrays.npz", allow_pickle=False) as arrays:
        if not np.array_equal(arrays["csv_rows"], selected_rows):
            raise RuntimeError(f"Attack subset mismatch for {name}.")
        if not np.array_equal(audit["csv_row_zero_based"].to_numpy(), selected_rows):
            raise RuntimeError(f"Audit row mismatch for {name}.")
        if not np.all(arrays["y"] == 1):
            raise RuntimeError("The attack subset must contain malicious records only.")
        changed = np.any(arrays["x_adv"] != arrays["x_clean"], axis=1)
    accepted = audit["valid"].to_numpy(dtype=bool) & audit["within_budget"].to_numpy(dtype=bool)
    initial = audit["clean_prediction"].to_numpy() == 1
    final = np.where(accepted, audit["adversarial_prediction"], audit["clean_prediction"])
    new_evasions = int((initial & (final == 0)).sum())
    if new_evasions != attack["newly_evaded"] or len(audit) != PILOT_RECORDS:
        raise RuntimeError(f"Saved attack counts disagree for {name}.")
    rows.append({"model": name, "attacked_malicious": len(audit),
                 "initially_detected": int(initial.sum()), "already_missed": int((~initial).sum()),
                 "returned_changed_rows": int(changed.sum()),
                 "changed_valid_in_budget": int((changed & accepted).sum()),
                 "new_evasions": new_evasions,
                 "new_evasion_percent_of_initially_detected": 100 * new_evasions / int(initial.sum()) if initial.any() else None,
                 "malicious_recall_after_attack_percent": 100 * float((final == 1).mean()),
                 "maximum_returned_scaled_l2_distance": float(audit["distance_l2_scaled"].max()),
                 "attack_wall_seconds": attack["attack_wall_seconds"]})
    print(f"{name}: {int(changed.sum())}/{len(audit)} returned rows changed; "
          f"{new_evasions}/{int(initial.sum())} initially detected records newly evaded.")
    if not changed.any():
        print("CAA returned the original rows. This does not establish robustness: no successful changed candidate was returned under this search budget.")

diagnostics = pd.DataFrame(rows)
diagnostics.to_csv(PILOT_DIR / "pilot_diagnostics.csv", index=False)
(PILOT_DIR / "pilot_checks.json").write_text(json.dumps({
    "clean_reference_checks": reference_checks,
    "same_selected_csv_rows_for_both_models": True,
    "attack_audit_counts_verified": True,
    "settings": report["settings"],
    "evaluation_device": report["device"],
    "scope": "Paired subset pilot; not a full-budget robust-score reproduction or live-traffic result."
}, indent=2) + "\n")
display(diagnostics)
print("Selected original CSV rows (zero-based):", selected_rows.tolist())
print("Full prediction files, candidate arrays and row audits are included in the results ZIP.")


## 6. Save and download the evidence

The ZIP includes results, timings, selected rows, candidate arrays, logs, hardware information, environment versions and a snapshot of the executed baseline scripts. It excludes the large source dataset and environment. Optional Drive copying prompts for authorization only when enabled.

**Share this ZIP**, even if no new evasions were found. If a cell fails, run this export cell separately to preserve partial logs. Colab runtime storage is temporary.


In [ ]:
EXPORTS = ROOT / "exports"
EXPORTS.mkdir(exist_ok=True)
export_base = EXPORTS / unique_name(SESSION.name + "_results")
archive = Path(shutil.make_archive(str(export_base), "zip", root_dir=SESSION.parent, base_dir=SESSION.name))
print("Results ZIP:", archive)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/MyPhDThesis/results")
    destination.mkdir(parents=True, exist_ok=True)
    target = destination / archive.name
    if target.exists():
        raise FileExistsError(f"Preserving existing archive: {target}")
    shutil.copy2(archive, target)
    print("Copied results to Google Drive:", target)
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        from IPython.display import FileLink
        display(FileLink(str(archive)))
    else:
        files.download(str(archive))


## What this experiment can establish

This pilot checks a larger bounded attack run and measures its cost. Increasing both sample size and search effort relative to notebook 01 is useful for execution planning, but it does not isolate the effect of either change. Clean accuracy is evaluated on the full test partition; attack recall concerns only the 32 selected malicious records.

Do not interpret zero new evasions as proof of security, or this feature-space experiment as real-world packet-level IDS performance. Full-budget evaluation and capability-aware training remain later stages. The checkpoint-compatible scaler still follows the upstream all-feature-rows convention; new training needs train-only preprocessing and separate development/test data.
